# Transcription — Pathway B: Meta MMS

For languages covered by Meta's [Massively Multilingual Speech (MMS)](https://ai.meta.com/blog/multilingual-model-speech-recognition/) `mms-1b-all` model that are not supported by Whisper.

**Run the language check cell (Section 1) before loading the model** — MMS marketing claims 1,100+ languages, but coverage of North American Indigenous languages is uneven in practice.

## Notable Indigenous languages in MMS (not in Whisper)

| Language | Code | Notes |
|---|---|---|
| Plains Cree | `crk-script_latin`, `crk-script_syllabics` | Syllabics support is significant |
| Ojibwe | `ojb-script_latin`, `ojb-script_syllabics` | Syllabics support |
| Mi'kmaq | `miq` | |
| O'odham (Tohono O'odham) | `ood` | |
| Gwich'in | `gwi` | |
| Dogrib / Tłı̨chǫ | `dgr` | |
| Guaraní | `grn` | |
| Rapanui | `rap` | |
| K'iche' | `quc-dialect_central`, `quc-dialect_east`, `quc-dialect_north` | |
| Kaqchikel | `cak-dialect_central` + 5 other dialects | |
| Mam | `mam-dialect_central` + 3 other dialects | |
| Quechua (multiple) | `quy`, `quz`, `qub`, `qve`, `qvh`, and others | |

**Absent from both Whisper and MMS:** Navajo, Cherokee, Hawaiian, Lakota/Dakota, and many others — use [02c_transcribe_manual.ipynb](02c_transcribe_manual.ipynb) for those.

## A note on MMS training data provenance

For many Indigenous languages, MMS training data was sourced primarily from Bible translations scraped without community consent. This has two practical consequences: output quality may be higher for religious vocabulary than everyday speech, and the model reflects that specific register. Review output carefully with fluent speakers.

All transcription runs locally. No audio leaves your machine.


## 0. Environment Check

```bash
uv sync --extra dev && source .venv/bin/activate
```

The `mms-1b-all` model is ~4 GB and will be downloaded to `~/.cache/huggingface/` on first run.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

issues = []
for pkg, mod in [("transformers", "transformers"), ("torch", "torch"),
                 ("librosa", "librosa"), ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        __import__(mod)
    except Exception as e:
        issues.append(f"{pkg}: {e}")
if issues:
    for i in issues: print(i)
    print("\nRun: uv sync --extra dev")
else:
    print("All dependencies available.")

## 1. Check Language Availability

This cell loads only the processor (lightweight — no 4 GB model download) and checks whether your language code is in the model.

In [ ]:
from transformers import AutoProcessor

MMS_LANGUAGE = "eng"  # Set your language code here

_proc = AutoProcessor.from_pretrained("facebook/mms-1b-all")
available = set(_proc.tokenizer.vocab.keys())

if MMS_LANGUAGE in available:
    print(f"'{MMS_LANGUAGE}' is available. Proceed to Section 2.")
else:
    print(f"'{MMS_LANGUAGE}' is NOT in this model.")
    print("Use 02c_transcribe_manual.ipynb instead.")
    matches = [k for k in sorted(available) if MMS_LANGUAGE.split("-")[0] in k]
    if matches:
        print(f"\nPossible related codes: {matches}")

## 2. Configure Paths

In [ ]:
INPUT_DIR  = "../test_data/wavs_export_audacity/"
OUTPUT_CSV = "../test_data/transcripts_mms.csv"

## 3. Load Model

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, AutoProcessor

model_id = "facebook/mms-1b-all"
print("Loading MMS model (downloads ~4 GB on first run)...")
processor = AutoProcessor.from_pretrained(model_id)
mms_model = Wav2Vec2ForCTC.from_pretrained(model_id)

processor.tokenizer.set_target_lang(MMS_LANGUAGE)
mms_model.load_adapter(MMS_LANGUAGE)
mms_model.eval()
print(f"Language adapter '{MMS_LANGUAGE}' loaded.")

## 4. Transcribe

In [ ]:
import os, csv
import librosa
from tqdm import tqdm

wav_files = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".wav"))
results   = []

for filename in tqdm(wav_files, desc="Transcribing (MMS)"):
    file_id  = os.path.splitext(filename)[0]
    filepath = os.path.join(INPUT_DIR, filename)

    waveform, _ = librosa.load(filepath, sr=16000, mono=True)
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt")

    with torch.no_grad():
        logits = mms_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcript = processor.decode(predicted_ids[0])
    results.append({"file_id": file_id, "transcript": transcript})
    print(f"  {file_id}: {transcript}")

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "transcript"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved to {OUTPUT_CSV}")

## 5. Review and Correct

MMS output requires careful human review. For languages where training data was primarily Bible translations, expect errors in everyday vocabulary, proper nouns, and place names.

Correct the `transcript` column to exactly match what was spoken. Preserve orthography exactly — syllabics characters, diacritics, and special characters are part of the language, not formatting.

Set `knowledge_keeper_reviewed = true` in the metadata after a fluent speaker has confirmed each transcript.

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f"{len(df)} transcripts")
df

## Next Steps

- Analyze audio quality: [03_snr_quality_analysis.ipynb](03_snr_quality_analysis.ipynb)
- Export final dataset: [05_export_ljspeech.ipynb](05_export_ljspeech.ipynb)